# SARIMA Testing on Multiple Products
This notebook tests SARIMA models on 10 selected products from the dataset.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import itertools
from datetime import datetime

# Set random seed for reproducibility
np.random.seed(42)

## Load Data and Select Products

In [ ]:
# Load dataset
df = pd.read_feather('../dataset/data_andre.feather')
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nTotal unique products: {df['item_id'].nunique()}")
print(f"Total unique stores: {df['store_id'].nunique()}")

In [ ]:
# Select 10 products with good data consistency
# Find products that have data across all time periods
product_counts = df.groupby('item_id').size().sort_values(ascending=False)
print(f"Top 20 products by number of records:")
print(product_counts.head(20))

# Select top 10 products
selected_products = product_counts.head(10).index.tolist()
print(f"\nSelected 10 products: {selected_products}")

In [ ]:
# Prepare data for each product - aggregate by date and item_id
df['date'] = pd.to_datetime(df['date'])
df_agg = df.groupby(['date', 'item_id'])['value'].sum().reset_index()
df_agg = df_agg.sort_values('date')

# Create a dictionary with time series for each product
product_series = {}
for item_id in selected_products:
    product_data = df_agg[df_agg['item_id'] == item_id].set_index('date')['value']
    product_series[item_id] = product_data
    print(f"Product {item_id}: {len(product_data)} data points")

## Define SARIMA Functions

In [ ]:
def grid_search_sarima(data, train_size=500, val_size=100, forecast_window=161, 
                       p_range=range(0, 3), d_range=range(0, 2), q_range=range(0, 3),
                       P_range=range(0, 2), D_range=range(0, 2), Q_range=range(0, 2),
                       s=12, verbose=False):
    """
    Grid search for optimal SARIMA parameters
    """
    # Ensure we have enough data
    total_required = train_size + val_size + forecast_window
    if len(data) < total_required:
        return None, None, None, f"Insufficient data: {len(data)} < {total_required}"
    
    total_train_val = train_size + val_size
    train = data[-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = data[-(val_size+forecast_window):-forecast_window].values
    test = data[-forecast_window:].values
    
    best_mae = float('inf')
    best_params = None
    best_seasonal_params = None
    results = []
    
    total_combinations = (len(p_range) * len(d_range) * len(q_range) * 
                         len(P_range) * len(D_range) * len(Q_range))
    if verbose:
        print(f"Testing {total_combinations} SARIMA parameter combinations...")
    
    count = 0
    for p, d, q in itertools.product(p_range, d_range, q_range):
        for P, D, Q in itertools.product(P_range, D_range, Q_range):
            try:
                count += 1
                if verbose and count % 20 == 0:
                    print(f"  Tested {count}/{total_combinations}...")
                
                # Fit ARIMA model (SARIMA with s=seasonal period)
                model = ARIMA(train, order=(p, d, q), 
                             seasonal_order=(P, D, Q, s),
                             disp=False)
                fitted_model = model.fit()
                
                # Predict on validation set
                val_predictions = fitted_model.get_forecast(steps=len(val)).predicted_mean.values
                
                # Calculate metrics
                mae = mean_absolute_error(val, val_predictions)
                rmse = np.sqrt(mean_squared_error(val, val_predictions))
                mape = mean_absolute_percentage_error(val, val_predictions) if (val != 0).any() else 0
                
                results.append({
                    'params': (p, d, q),
                    'seasonal_params': (P, D, Q),
                    'MAE': mae,
                    'RMSE': rmse,
                    'MAPE': mape
                })
                
                if mae < best_mae:
                    best_mae = mae
                    best_params = (p, d, q)
                    best_seasonal_params = (P, D, Q)
                    
            except Exception as e:
                pass  # Skip invalid parameter combinations
    
    if verbose:
        print(f"Best parameters found: {best_params} x {best_seasonal_params}")
    
    return results, best_params, best_seasonal_params, None

In [ ]:
def fit_and_evaluate_sarima(data, p, d, q, P, D, Q, s=12, train_size=500, val_size=100, forecast_window=161):
    """
    Fit SARIMA model with given parameters and evaluate on test set
    """
    total_train_val = train_size + val_size
    
    # Check data length
    if len(data) < total_train_val + forecast_window:
        return None, None
    
    train = data[-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = data[-(val_size+forecast_window):-forecast_window].values
    test = data[-forecast_window:].values
    
    try:
        # Train on combined train+val data
        full_train = np.concatenate([train, val])
        model = ARIMA(full_train, order=(p, d, q), seasonal_order=(P, D, Q, s))
        fitted_model = model.fit()
        
        # Make predictions on test set
        test_predictions = fitted_model.get_forecast(steps=len(test)).predicted_mean.values
        
        # Calculate metrics
        mse = mean_squared_error(test, test_predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(test, test_predictions)
        mape = mean_absolute_percentage_error(test, test_predictions) if (test != 0).any() else 0
        
        return {
            'test_predictions': test_predictions,
            'test_actual': test,
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE': mape
        }, fitted_model
        
    except Exception as e:
        return None, None

## Run SARIMA Grid Search on All 10 Products

In [ ]:
# Run grid search for each product
all_results = {}
best_models = {}

for idx, product_id in enumerate(selected_products, 1):
    print(f"\n{'='*60}")
    print(f"Product {idx}/10: Item ID {product_id}")
    print(f"{'='*60}")
    
    series = product_series[product_id]
    
    # Run grid search
    results, best_p, best_season, error = grid_search_sarima(
        series.values,
        train_size=300,
        val_size=50,
        forecast_window=100,
        p_range=range(0, 3),
        d_range=range(0, 2),
        q_range=range(0, 3),
        P_range=range(0, 2),
        D_range=range(0, 2),
        Q_range=range(0, 2),
        verbose=True
    )
    
    if error:
        print(f"  ❌ Error: {error}")
        continue
    
    # Evaluate best model on test set
    p, d, q = best_p
    P, D, Q = best_season
    
    eval_results, fitted_model = fit_and_evaluate_sarima(
        series.values,
        p, d, q, P, D, Q,
        s=12,
        train_size=300,
        val_size=50,
        forecast_window=100
    )
    
    if eval_results is None:
        print(f"  ❌ Could not fit model on test data")
        continue
    
    # Store results
    all_results[product_id] = {
        'best_params': best_p,
        'best_seasonal_params': best_season,
        'metrics': eval_results,
        'grid_search_results': pd.DataFrame(results)
    }
    best_models[product_id] = fitted_model
    
    # Print results
    print(f"  Best SARIMA: ({p},{d},{q})x({P},{D},{Q},12)")
    print(f"  Test Metrics:")
    print(f"    RMSE: {eval_results['RMSE']:.6f}")
    print(f"    MAE:  {eval_results['MAE']:.6f}")
    print(f"    MAPE: {eval_results['MAPE']:.6f}")

## Summary and Comparison

In [ ]:
# Create summary dataframe
summary_data = []
for product_id, result in all_results.items():
    p, d, q = result['best_params']
    P, D, Q = result['best_seasonal_params']
    metrics = result['metrics']
    
    summary_data.append({
        'Product_ID': product_id,
        'SARIMA_Order': f"({p},{d},{q})x({P},{D},{Q})",
        'RMSE': metrics['RMSE'],
        'MAE': metrics['MAE'],
        'MAPE': metrics['MAPE']
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("SARIMA Results Summary - All 10 Products")
print("="*80)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv('sarima_results_summary.csv', index=False)
print(f"\nSummary saved to: sarima_results_summary.csv")

In [ ]:
# Statistics
print("\n" + "="*80)
print("Overall Statistics")
print("="*80)
print(f"Average RMSE: {summary_df['RMSE'].mean():.6f}")
print(f"Average MAE:  {summary_df['MAE'].mean():.6f}")
print(f"Average MAPE: {summary_df['MAPE'].mean():.6f}")
print(f"\nBest RMSE: {summary_df['RMSE'].min():.6f} (Product {summary_df.loc[summary_df['RMSE'].idxmin(), 'Product_ID']})")
print(f"Worst RMSE: {summary_df['RMSE'].max():.6f} (Product {summary_df.loc[summary_df['RMSE'].idxmax(), 'Product_ID']})")

## Visualization - Predictions for Each Product

In [ ]:
# Plot predictions for all products
fig, axes = plt.subplots(5, 2, figsize=(16, 16))
axes = axes.flatten()

for idx, (product_id, result) in enumerate(all_results.items()):
    ax = axes[idx]
    
    test_actual = result['metrics']['test_actual']
    test_predictions = result['metrics']['test_predictions']
    p, d, q = result['best_params']
    P, D, Q = result['best_seasonal_params']
    rmse = result['metrics']['RMSE']
    
    ax.plot(test_actual, label='Actual', linewidth=2, marker='o', markersize=3)
    ax.plot(test_predictions, label='SARIMA Prediction', linewidth=2, linestyle='--', marker='s', markersize=3)
    ax.set_title(f'Product {product_id}\nSARIMA({p},{d},{q})x({P},{D},{Q},12) | RMSE: {rmse:.4f}')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sarima_predictions_all_products.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: sarima_predictions_all_products.png")

## Comparison Chart

In [ ]:
# Create comparison charts
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RMSE comparison
axes[0].bar(range(len(summary_df)), summary_df['RMSE'])
axes[0].set_xlabel('Product Index')
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE Comparison Across Products')
axes[0].set_xticks(range(len(summary_df)))
axes[0].set_xticklabels(summary_df['Product_ID'], rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# MAE comparison
axes[1].bar(range(len(summary_df)), summary_df['MAE'], color='orange')
axes[1].set_xlabel('Product Index')
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE Comparison Across Products')
axes[1].set_xticks(range(len(summary_df)))
axes[1].set_xticklabels(summary_df['Product_ID'], rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# MAPE comparison
axes[2].bar(range(len(summary_df)), summary_df['MAPE'], color='green')
axes[2].set_xlabel('Product Index')
axes[2].set_ylabel('MAPE')
axes[2].set_title('MAPE Comparison Across Products')
axes[2].set_xticks(range(len(summary_df)))
axes[2].set_xticklabels(summary_df['Product_ID'], rotation=45)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('sarima_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: sarima_metrics_comparison.png")